# 01 - NVDA EDA

Goal: understand price, return, risk, and feature relationships before modeling.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from nvda_rl.data_downloader import load_prices
from nvda_rl.features import add_market_features

sns.set_theme(style="whitegrid", context="notebook")

## Load saved data

This notebook does not download data. Run `scripts/download_data.py` first when the CSV needs refreshing.

In [ ]:
raw_path = PROJECT_ROOT / "data" / "raw" / "nvda_daily.csv"
prices = load_prices(raw_path)
print(f"Rows: {len(prices):,}")
print(f"Range: {prices['date'].min().date()} to {prices['date'].max().date()}")
prices.head()

## Build features

`add_market_features()` creates returns, volatility, momentum, volume z-score, drawdown, RSI, MACD, moving-average gaps, and ATR.

In [ ]:
data = add_market_features(prices)
data.to_csv(PROJECT_ROOT / "data" / "processed" / "nvda_features.csv", index=False)

feature_preview = [
    "date", "adj_close", "return", "volatility_20d", "momentum_20d",
    "rsi_14", "macd_hist", "sma_50_gap", "atr_14_pct", "drawdown"
]
data[feature_preview].head()

RSI shows stretched up/down moves, MACD shows trend change, and ATR shows daily range risk.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(data["date"], data["adj_close"], color="tab:green")
axes[0].set_title("NVDA adjusted close")
axes[0].set_ylabel("Price")

axes[1].fill_between(data["date"], data["drawdown"], 0, color="tab:red", alpha=0.35)
axes[1].set_title("Drawdown from running high")
axes[1].set_ylabel("Drawdown")
plt.tight_layout()
plt.show()

After feature warmup, NVDA grew from about $0.41 to $198.45. That huge gain still came with a max drawdown near -66%, so risk control matters.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(data["return"], bins=80, kde=True, ax=axes[0], color="tab:blue")
axes[0].set_title("Daily returns")
sns.histplot(data["volatility_20d"], bins=60, kde=True, ax=axes[1], color="tab:orange")
axes[1].set_title("20-day volatility")
sns.histplot(data["rsi_14"], bins=60, kde=True, ax=axes[2], color="tab:purple")
axes[2].set_title("RSI")
plt.tight_layout()
plt.show()

data[["return", "volatility_20d", "momentum_20d", "rsi_14", "atr_14_pct", "drawdown"]].describe().T

Average daily return is about 0.19%, while daily volatility is about 2.87%. Single-day moves ranged from about -18.8% to +29.8%, so the tails are meaningful.

## Correlations

Correlations show which features move together and which may add new information.

In [ ]:
corr_cols = [
    "return", "volatility_20d", "momentum_5d", "momentum_20d",
    "volume_z_20d", "drawdown", "rsi_14", "macd_hist",
    "sma_20_gap", "sma_50_gap", "atr_14_pct"
]
corr = data[corr_cols].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f", linewidths=0.5)
plt.title("Feature correlation heatmap")
plt.tight_layout()
plt.show()

corr["return"].sort_values(ascending=False)

The strongest return relationship is with 5-day momentum at about 0.44 correlation. ATR is slightly negative, which suggests high range-risk days are not automatically good return days.

In [ ]:
monthly = data.set_index("date")["return"].resample("M").agg(["mean", "std"])
monthly.columns = ["monthly_avg_daily_return", "monthly_daily_volatility"]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
monthly["monthly_avg_daily_return"].plot(ax=axes[0], color="tab:blue")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Monthly average daily return")
monthly["monthly_daily_volatility"].plot(ax=axes[1], color="tab:red")
axes[1].set_title("Monthly daily volatility")
plt.tight_layout()
plt.show()

Volatility clearly clusters in certain periods. A trading rule trained on calm periods may behave poorly in stress periods.